In [1]:
import pandas as pd
df = pd.read_csv("athlete_events.csv")
# only summer Olympics
df_summer = df[df["Season"].str.lower() == "summer"].copy()
# data cleanup Medal
df_summer["Medal"] = df_summer["Medal"].replace({"NA": pd.NA})
medals = df_summer[df_summer["Medal"].notna()].copy()
# Counted medals by NOC, Team, Sport
agg = (
    medals.groupby(["NOC", "Team", "Sport"])["Medal"]
    .value_counts()
    .unstack(fill_value=0)
    .reset_index()
)
# bronze/silver/gold may not exist
for col in ["Gold", "Silver", "Bronze"]:
    if col not in agg.columns:
        agg[col] = 0

agg["Total"] = agg[["Gold", "Silver", "Bronze"]].sum(axis=1)

# Gold → Total → Silver → Bronze → Sport (tie-break)
agg_sorted = agg.sort_values(
    by=["NOC", "Gold", "Total", "Silver", "Bronze", "Sport"],
    ascending=[True, False, False, False, False, True]
)

# choose first sport per NOC
iconic = agg_sorted.groupby("NOC", as_index=False).first()
iconic = iconic[["NOC", "Team", "Sport", "Gold", "Silver", "Bronze", "Total"]]
iconic.to_csv("iconic_sport_per_country.csv", index=False)
print("DONE! File saved as iconic_sport_per_country.csv")
print(iconic.head())


DONE! File saved as iconic_sport_per_country.csv
Medal  NOC                  Team      Sport  Gold  Silver  Bronze  Total
0      AFG           Afghanistan  Taekwondo     0       0       2      2
1      AHO  Netherlands Antilles    Sailing     0       1       0      1
2      ALG               Algeria  Athletics     4       3       2      9
3      ANZ           Australasia      Rugby    15       0       0     15
4      ARG             Argentina   Football    34      34       0     68
